In [1]:
import torch
import pandas as pd
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_path = "/content/drive/MyDrive/Models/Arabert"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

id2label = model.config.id2label


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [2]:
def predict_severity(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    probs = F.softmax(outputs.logits, dim=1).squeeze()

    predicted_idx = torch.argmax(probs).item()
    predicted_label = id2label[predicted_idx]

    return predicted_label, probs.cpu().numpy()


In [3]:
df = pd.read_excel("/content/drive/MyDrive/Datasets/Balanced/Train.xlsx")

df["Predicted class"] = ""
df["class1 prob"] = 0.0
df["class2 prob"] = 0.0
df["class3 prob"] = 0.0

In [5]:
from tqdm.auto import tqdm

for idx, row in tqdm(df.iterrows()):
    question = str(row["Question"])

    label, probs = predict_severity(question)

    df.at[idx, "Predicted class"] = label
    df.at[idx, "class1 prob"] = probs[0]
    df.at[idx, "class2 prob"] = probs[1]
    df.at[idx, "class3 prob"] = probs[2]

0it [00:00, ?it/s]

In [6]:
df.to_excel(
    "/content/drive/MyDrive/Datasets/Severity_Train.xlsx",
    index=False
)